In [1]:
import ebird

print("Loaded from:", ebird.__file__)
print("fetch_region_observations:", hasattr(ebird, "fetch_region_observations"))
print("save_raw_observations:", hasattr(ebird, "save_raw_observations"))
print("REGIONS:", hasattr(ebird, "REGIONS"))

Loaded from: c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\research\src\acquisition\ebird.py
fetch_region_observations: True
save_raw_observations: True
REGIONS: True


In [2]:
import sys
from pathlib import Path

ACQUISITION_DIR = Path.cwd()

if str(ACQUISITION_DIR) not in sys.path:
    sys.path.insert(0, str(ACQUISITION_DIR))

from xenocanto import (
    fetch_metadata,
    save_raw_metadata,
    download_audio_sample,
    A1_TARGETS,
)

from ebird import (
    fetch_region_observations,
    save_raw_observations,
    REGIONS,
)

print("Xeno-canto targets:", A1_TARGETS)
print("eBird regions:", REGIONS)

Xeno-canto targets: {'CA': 100, 'AZ': 75, 'TX': 100}
eBird regions: {'CA': 'US-CA', 'AZ': 'US-AZ', 'TX': 'US-TX'}


In [3]:
for state, target in A1_TARGETS.items():
    recs = fetch_metadata(state)
    save_raw_metadata(recs, state)
    download_audio_sample(
        recs,
        n=target,
        state=state
    )

[CA] Page 1/110 — 100 recordings (411 species total)
[CA] Page 2/110 — 100 recordings (411 species total)
[CA] Page 3/110 — 100 recordings (411 species total)
[CA] Saved 300 raw records to:
    C:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\raw\xenocanto\xc_metadata_raw_CA.json
[CA] Downloaded=100, Skipped=0, Failed=0
[AZ] Page 1/106 — 100 recordings (345 species total)
[AZ] Page 2/106 — 100 recordings (345 species total)
[AZ] Page 3/106 — 100 recordings (345 species total)
[AZ] Saved 300 raw records to:
    C:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\raw\xenocanto\xc_metadata_raw_AZ.json
[AZ] Downloaded=75, Skipped=0, Failed=0
[TX] Page 1/30 — 100 recordings (368 species total)
[TX] Page 2/30 — 100 recordings (368 species total)
[TX] Page 3/30 — 100 recordings (368 species total)
[TX] Saved 300 raw records to:
    C:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\raw\xenocanto\xc_metadata_raw_TX.json
[TX] Downloaded=100, Skip

In [4]:
for state, region_code in REGIONS.items():
    obs = fetch_region_observations(region_code)
    save_raw_observations(obs, state)

[CA] Saved 495 raw eBird observations to:
    C:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\raw\ebird\ebird_observations_raw_CA.json
[AZ] Saved 373 raw eBird observations to:
    C:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\raw\ebird\ebird_observations_raw_AZ.json
[TX] Saved 458 raw eBird observations to:
    C:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\raw\ebird\ebird_observations_raw_TX.json


In [5]:
print("A1 DATA ACQUISITION SUMMARY")
print("=" * 50)

print("\nXeno-canto")
print("-" * 50)

for state, target in A1_TARGETS.items():
    print(f"{state}: target = {target} audio files")

print("\nTotal Xeno-canto audio target:",
      sum(A1_TARGETS.values()))

print("\neBird")
print("-" * 50)

for state, region_code in REGIONS.items():
    print(f"{state}: {region_code}")

print("\nRaw acquisition completed successfully.")

A1 DATA ACQUISITION SUMMARY

Xeno-canto
--------------------------------------------------
CA: target = 100 audio files
AZ: target = 75 audio files
TX: target = 100 audio files

Total Xeno-canto audio target: 275

eBird
--------------------------------------------------
CA: US-CA
AZ: US-AZ
TX: US-TX

Raw acquisition completed successfully.


In [7]:
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[2]
RAW_XC_DIR = PROJECT_ROOT / "data" / "raw" / "xenocanto"

for state in A1_TARGETS:

    metadata_path = RAW_XC_DIR / f"xc_metadata_raw_{state}.json"

    with open(metadata_path, encoding="utf-8") as f:
        recs = json.load(f)[:A1_TARGETS[state]]

    species = {
        r["en"]
        for r in recs
        if r.get("en")
    }

    print(
        f"{state}: "
        f"{len(recs)} recordings, "
        f"{len(species)} unique species"
    )

CA: 100 recordings, 6 unique species
AZ: 75 recordings, 16 unique species
TX: 100 recordings, 15 unique species


In [8]:
print("A1 Xeno-canto sample")
print("=" * 40)

sample_species = {
    "CA": 6,
    "AZ": 16,
    "TX": 15,
}

for state, target in A1_TARGETS.items():
    print(
        f"{state}: "
        f"{target} recordings | "
        f"{sample_species[state]} unique species"
    )

print(f"\nTotal recordings: {sum(A1_TARGETS.values())}")
print("Audio download failures: 0")
print("Restricted/skipped recordings: 0")

A1 Xeno-canto sample
CA: 100 recordings | 6 unique species
AZ: 75 recordings | 16 unique species
TX: 100 recordings | 15 unique species

Total recordings: 275
Audio download failures: 0
Restricted/skipped recordings: 0


In [9]:
for state in A1_TARGETS:
    with open(RAW_XC_DIR / f"xc_metadata_raw_{state}.json", encoding="utf-8") as f:
        recs = json.load(f)
    species = {r["en"] for r in recs if r.get("en")}
    print(f"{state}: {len(recs)} total metadata records, {len(species)} unique species available")

CA: 300 total metadata records, 25 unique species available
AZ: 300 total metadata records, 21 unique species available
TX: 300 total metadata records, 44 unique species available


In [11]:
import pandas as pd
for state in A1_TARGETS:

    with open(
        RAW_XC_DIR / f"xc_metadata_raw_{state}.json",
        encoding="utf-8"
    ) as f:
        recs = json.load(f)

    species_counts = (
        pd.Series([r["en"] for r in recs if r.get("en")])
        .value_counts()
    )

    print(f"\n{state}")
    print("=" * 50)
    print(species_counts)


CA
Mountain Quail                 65
Snow Goose                     44
California Quail               37
Ross's Goose                   30
Mallard                        19
Cackling Goose                 16
American Wigeon                11
Green-winged Teal               9
Greater White-fronted Goose     8
Brant Goose                     8
Gadwall                         7
Canada Goose                    6
Tundra Swan                     5
Ruddy Duck                      5
Egyptian Goose                  5
Cinnamon Teal                   5
Wood Duck                       5
Northern Pintail                3
Northern Shoveler               3
Redhead                         2
Blue-winged Teal                2
Bufflehead                      2
Short-billed Gull               1
Surf Scoter                     1
Lesser Scaup                    1
Name: count, dtype: int64

AZ
Gambel's Quail                  99
Montezuma Quail                 51
Scaled Quail                    37
Wild Turkey

In [12]:
import random
from collections import defaultdict

In [13]:
RANDOM_SEED = 42


def diverse_sample(records, n, seed=42):
    """
    Select a reproducible species-diverse sample.

    Records are grouped by species, shuffled within species,
    and selected in round-robin fashion. This prioritizes
    species coverage before allocating additional recordings
    to species with larger candidate pools.
    """
    
    rng = random.Random(seed)

    by_species = defaultdict(list)

    for record in records:
        species = record.get("en")

        if species:
            by_species[species].append(record)

    # Shuffle recordings within each species
    for species_records in by_species.values():
        rng.shuffle(species_records)

    # Sort species names for deterministic ordering
    species_names = sorted(by_species.keys())

    selected = []
    position = {species: 0 for species in species_names}

    # Round-robin selection across species
    while len(selected) < n:

        added_this_round = False

        for species in species_names:

            if len(selected) >= n:
                break

            i = position[species]

            if i < len(by_species[species]):
                selected.append(by_species[species][i])
                position[species] += 1
                added_this_round = True

        if not added_this_round:
            break

    return selected

In [14]:
candidate_pools = {}

for state in A1_TARGETS:

    pool = fetch_metadata(
        state,
        max_pages=10
    )

    candidate_pools[state] = pool

    species_pool = {
        r["en"]
        for r in pool
        if r.get("en")
    }

    print(
        f"{state}: "
        f"{len(pool)} candidate records | "
        f"{len(species_pool)} species"
    )

[CA] Page 1/110 — 100 recordings (411 species total)
[CA] Page 2/110 — 100 recordings (411 species total)
[CA] Page 3/110 — 100 recordings (411 species total)
[CA] Page 4/110 — 100 recordings (411 species total)
[CA] Page 5/110 — 100 recordings (411 species total)
[CA] Page 6/110 — 100 recordings (411 species total)
[CA] Page 7/110 — 100 recordings (411 species total)
[CA] Page 8/110 — 100 recordings (411 species total)
[CA] Page 9/110 — 100 recordings (411 species total)
[CA] Page 10/110 — 100 recordings (411 species total)
CA: 1000 candidate records | 66 species
[AZ] Page 1/106 — 100 recordings (345 species total)
[AZ] Page 2/106 — 100 recordings (345 species total)
[AZ] Page 3/106 — 100 recordings (345 species total)
[AZ] Page 4/106 — 100 recordings (345 species total)
[AZ] Page 5/106 — 100 recordings (345 species total)
[AZ] Page 6/106 — 100 recordings (345 species total)
[AZ] Page 7/106 — 100 recordings (345 species total)
[AZ] Page 8/106 — 100 recordings (345 species total)
[AZ] 

In [15]:
selected_records = {}

for state, target in A1_TARGETS.items():

    selected = diverse_sample(
        candidate_pools[state],
        n=target,
        seed=RANDOM_SEED
    )

    selected_records[state] = selected

    selected_species = {
        r["en"]
        for r in selected
        if r.get("en")
    }

    print(
        f"{state}: "
        f"{len(selected)} selected recordings | "
        f"{len(selected_species)} species"
    )

CA: 100 selected recordings | 66 species
AZ: 75 selected recordings | 44 species
TX: 100 selected recordings | 100 species


In [16]:
for state, records in selected_records.items():

    counts = (
        pd.Series(
            [r["en"] for r in records if r.get("en")]
        )
        .value_counts()
    )

    print(f"\n{state}")
    print("=" * 60)
    print(f"Selected recordings: {len(records)}")
    print(f"Unique species: {len(counts)}")
    print("\nSpecies distribution:")
    print(counts)


CA
Selected recordings: 100
Unique species: 66

Species distribution:
African Collared Dove    2
Allen's Hummingbird      2
American Coot            2
American Wigeon          2
Anna's Hummingbird       2
                        ..
White-winged Dove        1
Wild Turkey              1
Wood Duck                1
Yellow Rail              1
Yellow-billed Cuckoo     1
Name: count, Length: 66, dtype: int64

AZ
Selected recordings: 75
Unique species: 44

Species distribution:
Allen's Hummingbird             2
American Wigeon                 2
Anna's Hummingbird              2
Band-tailed Pigeon              2
Berylline Hummingbird           2
Black-chinned Hummingbird       2
Blue-throated Mountaingem       2
Broad-billed Hummingbird        2
Buff-collared Nightjar          2
Broad-tailed Hummingbird        2
Cinnamon Teal                   2
Calliope Hummingbird            2
Rivoli's Hummingbird            2
Ring-necked Duck                2
Common Nighthawk                2
Common Poorwil

In [17]:
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[2]

SELECTED_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "xenocanto_selected"
)

SELECTED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Selected metadata directory:")
print(SELECTED_DIR)

Selected metadata directory:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\interim\xenocanto_selected


In [18]:
for state, records in selected_records.items():

    path = (
        SELECTED_DIR
        / f"xc_a1_selected_{state}.json"
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            records,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        f"{state}: "
        f"saved {len(records)} selected records"
    )

CA: saved 100 selected records
AZ: saved 75 selected records
TX: saved 100 selected records


In [19]:
manifest_rows = []

for state, records in selected_records.items():

    for record in records:

        manifest_rows.append({
            "state": state,
            "recording_id": record.get("id"),
            "common_name": record.get("en"),
            "scientific_name": (
                f"{record.get('gen', '')} "
                f"{record.get('sp', '')}"
            ).strip(),
            "quality": record.get("q"),
            "date": record.get("date"),
            "time": record.get("time"),
            "latitude": record.get("lat"),
            "longitude": record.get("lon"),
            "duration": record.get("length"),
            "audio_url": record.get("file"),
            "source_url": record.get("url"),
            "license": record.get("lic"),
            "selection_seed": RANDOM_SEED,
            "selection_method": "species_stratified_round_robin"
        })

selection_manifest = pd.DataFrame(manifest_rows)

MANIFEST_DIR = (
    PROJECT_ROOT
    / "data"
    / "manifests"
)

MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)

manifest_path = (
    MANIFEST_DIR
    / "a1_xenocanto_selection_manifest.csv"
)

selection_manifest.to_csv(
    manifest_path,
    index=False
)

print(
    f"Saved {len(selection_manifest)} records to:"
)
print(manifest_path)

Saved 275 records to:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\manifests\a1_xenocanto_selection_manifest.csv


In [20]:
selection_manifest.head()

,state,recording_id,common_name,scientific_name,quality,date,time,latitude,longitude,duration,audio_url,source_url,license,selection_seed,selection_method
0,CA,375492,African Collared Dove,Streptopelia roseogrisea,A,2017-06-14,15:00,33.922,-117.2617,0:05,https://xeno-canto.org/375492/download,https://xeno-canto.org/375492,https://creativecommons.org/licenses/by-nc-sa/...,42,species_stratified_round_robin
1,CA,952203,Allen's Hummingbird,Selasphorus sasin,A,2024-11-24,08:00,33.6104,-117.7392,0:38,https://xeno-canto.org/952203/download,https://xeno-canto.org/952203,https://creativecommons.org/licenses/by-nc-sa/...,42,species_stratified_round_robin
2,CA,452094,American Coot,Fulica americana,A,2018-01-30,12:56,33.2016,-115.597,0:18,https://xeno-canto.org/452094/download,https://xeno-canto.org/452094,https://creativecommons.org/licenses/by-nc-sa/...,42,species_stratified_round_robin
3,CA,172887,American Wigeon,Mareca americana,B,2012-03-23,09:54,32.564,-117.1256,0:11,https://xeno-canto.org/172887/download,https://xeno-canto.org/172887,https://creativecommons.org/licenses/by-nc-sa/...,42,species_stratified_round_robin
4,CA,449497,Anna's Hummingbird,Calypte anna,A,2018-10-21,07:46,32.686,-117.243,1:50,https://xeno-canto.org/449497/download,https://xeno-canto.org/449497,https://creativecommons.org/licenses/by-nc-sa/...,42,species_stratified_round_robin


In [21]:
print("Total selected:", len(selection_manifest))

print("\nBy state:")
print(selection_manifest["state"].value_counts())

print("\nDuplicate recording IDs:")
print(selection_manifest["recording_id"].duplicated().sum())

Total selected: 275

By state:
state
CA    100
TX    100
AZ     75
Name: count, dtype: int64

Duplicate recording IDs:
1


In [22]:
duplicates = selection_manifest[
    selection_manifest["recording_id"].duplicated(keep=False)
].sort_values("recording_id")

duplicates[
    [
        "state",
        "recording_id",
        "common_name",
        "scientific_name",
        "audio_url",
        "source_url"
    ]
]

,state,recording_id,common_name,scientific_name,audio_url,source_url
53,CA,121403,Snow Goose,Anser caerulescens,https://xeno-canto.org/121403/download,https://xeno-canto.org/121403
138,AZ,121403,Snow Goose,Anser caerulescens,https://xeno-canto.org/121403/download,https://xeno-canto.org/121403


In [23]:
print(
    "Number of unique recording IDs:",
    selection_manifest["recording_id"].nunique()
)

print(
    "Number of rows:",
    len(selection_manifest)
)

Number of unique recording IDs: 274
Number of rows: 275


In [24]:
for state, records in candidate_pools.items():

    ids = [
        str(r["id"])
        for r in records
        if r.get("id") is not None
    ]

    print(
        f"{state}: "
        f"{len(ids)} records | "
        f"{len(set(ids))} unique IDs | "
        f"{len(ids) - len(set(ids))} duplicates"
    )

CA: 1000 records | 1000 unique IDs | 0 duplicates
AZ: 1000 records | 1000 unique IDs | 0 duplicates
TX: 1000 records | 1000 unique IDs | 0 duplicates


In [25]:
deduplicated_pools = {}

for state, records in candidate_pools.items():

    seen_ids = set()
    unique_records = []

    for record in records:

        recording_id = record.get("id")

        if recording_id is None:
            continue

        recording_id = str(recording_id)

        if recording_id not in seen_ids:
            seen_ids.add(recording_id)
            unique_records.append(record)

    deduplicated_pools[state] = unique_records

    print(
        f"{state}: "
        f"{len(records)} original → "
        f"{len(unique_records)} unique"
    )

CA: 1000 original → 1000 unique
AZ: 1000 original → 1000 unique
TX: 1000 original → 1000 unique


In [26]:
selected_records = {}

for state, target in A1_TARGETS.items():

    selected = diverse_sample(
        deduplicated_pools[state],
        n=target,
        seed=RANDOM_SEED
    )

    selected_records[state] = selected

    selected_species = {
        r["en"]
        for r in selected
        if r.get("en")
    }

    print(
        f"{state}: "
        f"{len(selected)} selected recordings | "
        f"{len(selected_species)} species"
    )

CA: 100 selected recordings | 66 species
AZ: 75 selected recordings | 44 species
TX: 100 selected recordings | 100 species


In [27]:
selected_ids = []

for state, records in selected_records.items():

    selected_ids.extend(
        str(r["id"])
        for r in records
        if r.get("id") is not None
    )

print("Total selected:", len(selected_ids))
print("Unique IDs:", len(set(selected_ids)))
print("Duplicate IDs:", len(selected_ids) - len(set(selected_ids)))

Total selected: 275
Unique IDs: 274
Duplicate IDs: 1


In [28]:
for state, records in candidate_pools.items():

    ids = [
        str(r["id"])
        for r in records
        if r.get("id") is not None
    ]

    duplicate_ids = {
        recording_id
        for recording_id in ids
        if ids.count(recording_id) > 1
    }

    print(
        f"{state}: "
        f"{len(ids)} records | "
        f"{len(set(ids))} unique IDs | "
        f"{len(duplicate_ids)} duplicate IDs"
    )

CA: 1000 records | 1000 unique IDs | 0 duplicate IDs
AZ: 1000 records | 1000 unique IDs | 0 duplicate IDs
TX: 1000 records | 1000 unique IDs | 0 duplicate IDs


In [29]:
RANDOM_SEED = 42


def diverse_sample(records, n, seed=42):
    """
    Select a reproducible species-diverse sample with
    guaranteed unique Xeno-canto recording IDs.

    Records are grouped by species, shuffled within species,
    and selected in round-robin fashion. A recording ID can
    only be selected once.
    """

    rng = random.Random(seed)

    by_species = defaultdict(list)
    seen_ids = set()

    # Build species-specific pools while enforcing unique IDs
    for record in records:

        species = record.get("en")
        recording_id = record.get("id")

        if not species or recording_id is None:
            continue

        recording_id = str(recording_id)

        if recording_id in seen_ids:
            continue

        seen_ids.add(recording_id)
        by_species[species].append(record)

    # Shuffle within each species
    for species_records in by_species.values():
        rng.shuffle(species_records)

    species_names = sorted(by_species.keys())

    selected = []
    selected_ids = set()

    position = {
        species: 0
        for species in species_names
    }

    # Round-robin across species
    while len(selected) < n:

        added_this_round = False

        for species in species_names:

            if len(selected) >= n:
                break

            i = position[species]

            if i < len(by_species[species]):

                record = by_species[species][i]
                recording_id = str(record["id"])

                position[species] += 1

                # Extra safety check
                if recording_id in selected_ids:
                    continue

                selected.append(record)
                selected_ids.add(recording_id)
                added_this_round = True

        if not added_this_round:
            break

    return selected

In [30]:
selected_records = {}

for state, target in A1_TARGETS.items():

    selected = diverse_sample(
        candidate_pools[state],
        n=target,
        seed=RANDOM_SEED
    )

    selected_records[state] = selected

    selected_species = {
        r["en"]
        for r in selected
        if r.get("en")
    }

    print(
        f"{state}: "
        f"{len(selected)} selected recordings | "
        f"{len(selected_species)} species"
    )

CA: 100 selected recordings | 66 species
AZ: 75 selected recordings | 44 species
TX: 100 selected recordings | 100 species


In [31]:
selected_ids = []

for state, records in selected_records.items():

    selected_ids.extend(
        str(r["id"])
        for r in records
        if r.get("id") is not None
    )

print("Total selected:", len(selected_ids))
print("Unique IDs:", len(set(selected_ids)))
print(
    "Duplicate IDs:",
    len(selected_ids) - len(set(selected_ids))
)

Total selected: 275
Unique IDs: 274
Duplicate IDs: 1


In [32]:
from collections import Counter

selected_id_records = []

for state, records in selected_records.items():
    for record in records:
        selected_id_records.append({
            "state": state,
            "id": str(record["id"]),
            "species": record.get("en")
        })

id_counts = Counter(
    row["id"]
    for row in selected_id_records
)

duplicate_ids = {
    recording_id: count
    for recording_id, count in id_counts.items()
    if count > 1
}

print("Total selected:", len(selected_id_records))
print("Unique IDs:", len(id_counts))
print("Duplicate IDs:", duplicate_ids)

Total selected: 275
Unique IDs: 274
Duplicate IDs: {'121403': 2}


In [33]:
for duplicate_id in duplicate_ids:

    print(f"\nDUPLICATE ID: {duplicate_id}")

    for row in selected_id_records:
        if row["id"] == duplicate_id:
            print(row)


DUPLICATE ID: 121403
{'state': 'CA', 'id': '121403', 'species': 'Snow Goose'}
{'state': 'AZ', 'id': '121403', 'species': 'Snow Goose'}


In [34]:
global_seen_ids = set()
unique_candidate_pools = {}

for state in ["CA", "AZ", "TX"]:

    unique_records = []

    for record in candidate_pools[state]:

        recording_id = record.get("id")

        if recording_id is None:
            continue

        recording_id = str(recording_id)

        if recording_id in global_seen_ids:
            continue

        global_seen_ids.add(recording_id)
        unique_records.append(record)

    unique_candidate_pools[state] = unique_records

    print(
        f"{state}: "
        f"{len(candidate_pools[state])} original → "
        f"{len(unique_records)} globally unique"
    )

print(
    f"\nTotal globally unique candidate recordings: "
    f"{len(global_seen_ids)}"
)

CA: 1000 original → 1000 globally unique
AZ: 1000 original → 972 globally unique
TX: 1000 original → 1000 globally unique

Total globally unique candidate recordings: 2972


In [35]:
selected_records = {}

for state, target in A1_TARGETS.items():

    selected = diverse_sample(
        unique_candidate_pools[state],
        n=target,
        seed=RANDOM_SEED
    )

    selected_records[state] = selected

    selected_species = {
        r["en"]
        for r in selected
        if r.get("en")
    }

    print(
        f"{state}: "
        f"{len(selected)} selected recordings | "
        f"{len(selected_species)} species"
    )

CA: 100 selected recordings | 66 species
AZ: 75 selected recordings | 44 species
TX: 100 selected recordings | 100 species


In [36]:
selected_ids = []

for state, records in selected_records.items():

    selected_ids.extend(
        str(r["id"])
        for r in records
        if r.get("id") is not None
    )

print("Total selected:", len(selected_ids))
print("Unique IDs:", len(set(selected_ids)))
print(
    "Duplicate IDs:",
    len(selected_ids) - len(set(selected_ids))
)

Total selected: 275
Unique IDs: 275
Duplicate IDs: 0


In [37]:
for state, records in selected_records.items():

    species = {
        r["en"]
        for r in records
        if r.get("en")
    }

    print(
        f"{state}: "
        f"{len(records)} recordings | "
        f"{len(species)} species"
    )

CA: 100 recordings | 66 species
AZ: 75 recordings | 44 species
TX: 100 recordings | 100 species


In [38]:
for state, records in selected_records.items():

    path = (
        SELECTED_DIR
        / f"xc_a1_selected_{state}.json"
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            records,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        f"{state}: saved {len(records)} selected records"
    )

CA: saved 100 selected records
AZ: saved 75 selected records
TX: saved 100 selected records


In [39]:
manifest_rows = []

for state, records in selected_records.items():

    for record in records:

        manifest_rows.append({
            "state": state,
            "recording_id": record.get("id"),
            "common_name": record.get("en"),
            "scientific_name": (
                f"{record.get('gen', '')} "
                f"{record.get('sp', '')}"
            ).strip(),
            "quality": record.get("q"),
            "date": record.get("date"),
            "time": record.get("time"),
            "latitude": record.get("lat"),
            "longitude": record.get("lon"),
            "duration": record.get("length"),
            "audio_url": record.get("file"),
            "source_url": record.get("url"),
            "license": record.get("lic"),
            "selection_seed": RANDOM_SEED,
            "selection_method": "species_stratified_round_robin"
        })

selection_manifest = pd.DataFrame(manifest_rows)

manifest_path = (
    MANIFEST_DIR
    / "a1_xenocanto_selection_manifest.csv"
)

selection_manifest.to_csv(
    manifest_path,
    index=False
)

print(f"Saved {len(selection_manifest)} records")
print(manifest_path)

Saved 275 records
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\manifests\a1_xenocanto_selection_manifest.csv


In [40]:
print("Total rows:", len(selection_manifest))

print("\nBy state:")
print(selection_manifest["state"].value_counts())

print("\nUnique recording IDs:")
print(selection_manifest["recording_id"].nunique())

print("\nDuplicate recording IDs:")
print(
    selection_manifest["recording_id"]
    .duplicated()
    .sum()
)

Total rows: 275

By state:
state
CA    100
TX    100
AZ     75
Name: count, dtype: int64

Unique recording IDs:
275

Duplicate recording IDs:
0


In [41]:
SELECTED_AUDIO_DIR = (
    PROJECT_ROOT
    / "audio"
    / "validated"
)

SELECTED_AUDIO_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Selected audio directory:")
print(SELECTED_AUDIO_DIR)

Selected audio directory:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\audio\validated


In [42]:
download_results = {}

for state, records in selected_records.items():

    result = download_audio_sample(
        records,
        n=len(records),
        state=state,
        out_dir=SELECTED_AUDIO_DIR
    )

    download_results[state] = result

    print(f"\n{state}")
    print(result)

[CA] Downloaded=100, Skipped=0, Failed=0

CA
{'state': 'CA', 'requested': 100, 'downloaded': 100, 'skipped': 0, 'failed': 0}
[AZ] Downloaded=75, Skipped=0, Failed=0

AZ
{'state': 'AZ', 'requested': 75, 'downloaded': 75, 'skipped': 0, 'failed': 0}
[TX] Downloaded=100, Skipped=0, Failed=0

TX
{'state': 'TX', 'requested': 100, 'downloaded': 100, 'skipped': 0, 'failed': 0}


In [43]:
for state, records in selected_records.items():

    expected_ids = {
        str(r["id"])
        for r in records
    }

    downloaded_ids = {
        path.stem.replace(
            f"{state}_",
            ""
        )
        for path in SELECTED_AUDIO_DIR.glob(
            f"{state}_*.mp3"
        )
    }

    matched = expected_ids & downloaded_ids

    print(
        f"{state}: "
        f"{len(matched)}/{len(expected_ids)} "
        f"selected recordings downloaded"
    )

CA: 100/100 selected recordings downloaded
AZ: 75/75 selected recordings downloaded
TX: 100/100 selected recordings downloaded
